# 01 — Scraping de FBref

**Objetivo**: descargar stats reales de las 4 ligas para la herramienta.

**Ligas**:
- Liga MX (México)
- Brasileirão Série A (Brasil)
- Liga Profesional Argentina (Primera Argentina)
- La Liga 2 (Segunda España)

**Temporadas**: 2024-25 y 2025-26

**Tiempo estimado**: 15-30 minutos (pausas obligatorias para no ser baneados por FBref).

## ⚠️ Antes de ejecutar

1. Asegúrate de tener instalado:
   ```
   pip install fbrefdata pandas
   ```

2. Si una celda falla con `RateLimitError`, **espera 5 minutos y reintenta**.

3. Los datos descargados quedan en `data/raw/`. Si quieres re-descargar, borra esos CSV o usa `force_refresh=True`.

In [1]:
# Setup
import sys
from pathlib import Path

# añadir src al path (asumiendo que el notebook está en notebooks/)
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src import data_fetcher as df_mod
import pandas as pd
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 200)

print(f'Project root: {PROJECT_ROOT}')
print(f'Ligas configuradas: {list(df_mod.LEAGUES_FBREF.keys())}')

Project root: C:\Users\msanr\Downloads\rayados_scouting_lab_v3_bloque1\rayados_scouting_lab
Ligas configuradas: ['Liga MX', 'Brasileirao', 'Primera Argentina', 'Segunda Espanola']


## Paso 1: Verificar que fbrefdata reconoce las ligas

La librería `fbrefdata` por defecto soporta más ligas que `soccerdata`, pero hay que verificar que nuestras 4 están disponibles.

In [2]:
import fbrefdata as fd

available = fd.FBref.available_leagues()
print(f'Total ligas en fbrefdata: {len(available)}')
print()

# verificar las nuestras
necesarias = list(df_mod.LEAGUES_FBREF.values())
for liga in necesarias:
    estado = '✓' if liga in available else '✗'
    print(f'  {estado} {liga}')

faltantes = [l for l in necesarias if l not in available]
if faltantes:
    print(f'\n⚠️ Ligas no soportadas directamente: {faltantes}')
    print('Solución: añadirlas al config json (ver siguiente celda).')
else:
    print('\n✅ Todas las ligas disponibles. Continuar.')

[05/16/26 19:27:10] INFO     No custom league dict found. You need to select leagues in               ]8;id=936194;file://C:\Users\msanr\anaconda3\envs\rayados\Lib\site-packages\fbrefdata\_config.py\_config.py]8;;\:]8;id=897462;file://C:\Users\msanr\anaconda3\envs\rayados\Lib\site-packages\fbrefdata\_config.py#98\98]8;;\
                             C:\Users\msanr\fbrefdata\config\league_dict.json.                                     

Total ligas en fbrefdata: 1

  ✗ MEX-Liga MX
  ✗ BRA-Série A
  ✗ ARG-Liga Profesional Argentina
  ✗ ESP-La Liga 2

⚠️ Ligas no soportadas directamente: ['MEX-Liga MX', 'BRA-Série A', 'ARG-Liga Profesional Argentina', 'ESP-La Liga 2']
Solución: añadirlas al config json (ver siguiente celda).


### Si alguna liga NO está soportada

Hay que editar el archivo `league_dict.json` de fbrefdata. Su ubicación se imprime al importar la librería; en Windows suele ser:

```
C:\Users\<tu_usuario>\fbrefdata\config\league_dict.json
```

Añade entradas como:

```json
{
  "MEX-Liga MX": {
    "FBref": "Liga MX"
  },
  "BRA-Série A": {
    "FBref": "Série A"
  }
}
```

Y reinicia el kernel del notebook.

## Paso 2: Descargar stats de equipos (las 4 ligas)

Esto alimenta el **benchmark competitivo** de la app: Rayados vs últimos campeones.

Solo necesitamos Liga MX para benchmark, las otras 3 son para scouting de jugadores.

In [ ]:
# Solo Liga MX para el benchmark
from src.data_fetcher import fetch_fbref_team_stats

teams_ligamx = {}
for season in ['2425', '2526']:
    print(f'\n=== Liga MX {season} ===')
    teams_ligamx[season] = fetch_fbref_team_stats('Liga MX', season)

# Inspeccionar
print('\nColumnas devueltas (primer nivel):')
print(teams_ligamx['2526'].columns.get_level_values(0).unique().tolist())
teams_ligamx['2526'].head()

## Paso 3: Descargar stats de jugadores (las 4 ligas)

Esto alimenta el **scouting**. Es la descarga más larga (8 archivos: 4 ligas × 2 temporadas).

**Tiempo estimado**: 20 minutos.

In [ ]:
from src.data_fetcher import fetch_fbref_player_stats

players_data = {}
for liga in df_mod.LEAGUES_FBREF:
    players_data[liga] = {}
    for season in ['2425', '2526']:
        print(f'\n=== {liga} {season} ===')
        try:
            players_data[liga][season] = fetch_fbref_player_stats(liga, season)
            print(f'  ✓ {len(players_data[liga][season])} jugadores')
        except Exception as e:
            print(f'  ✗ FALLÓ: {e}')
            players_data[liga][season] = None

## Paso 4: Sanity checks rápidos

Verificar que los datos descargados tienen sentido antes de pasar al cleaning.

In [ ]:
for liga, seasons in players_data.items():
    df = seasons.get('2526')
    if df is None or df.empty:
        print(f'⚠️ {liga} 2526: SIN DATOS')
        continue

    n = len(df)
    # buscar columna de minutos (suele ser ('standard', 'Min') o similar)
    min_cols = [c for c in df.columns if 'Min' in str(c)]
    if min_cols:
        mean_min = df[min_cols[0]].mean()
    else:
        mean_min = 'N/A'
    print(f'{liga:20s} | {n:4d} jugadores | min promedio: {mean_min}')

print('\n✅ Si todas las ligas tienen 300+ jugadores y minutos promedio coherentes, continuar.')
print('   Si alguna tiene 0 o muy pocos jugadores, revisar el log de errores arriba.')

## Paso 5: Confirmación final

Listar lo descargado y guardado.

In [ ]:
raw_dir = PROJECT_ROOT / 'data' / 'raw'
archivos = sorted(raw_dir.glob('*.csv'))
print(f'Archivos en {raw_dir}:')
for f in archivos:
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:60s} {size_kb:8.1f} KB')

print(f'\nTotal: {len(archivos)} archivos')
print('\n✅ Listo. Continuar con notebook 02 (Transfermarkt) y luego 03 (cleaning).')